# Scaling Laws for Cognitive Alignment

This notebook accompanies **Section 5.5** of the LEVANTE-bench paper.

**Research question:** As VLMs get bigger, do they become more *human-like* in a predictable way?

**Motivation:**
- Observational Scaling Laws (Ruan et al., NeurIPS 2024) and Sloth (Polo et al., NeurIPS 2025) showed that LLM *benchmark performance* follows smooth sigmoidal curves as a function of compute — but only for accuracy, not for alignment with humans.
- DevBench (Tan et al., NeurIPS 2024) showed that OpenCLIP becomes more human-like over training — but only for a single model.
- We extend both lines: we test whether **cognitive alignment with children** scales predictably across **20 models from 7 families** (0.256B–200B parameters), measuring **three levels** of alignment.

**What's new vs. the existing paper plots:**
- The paper already has accuracy vs. log(params) per task (Fig. 1, David's plot) — but with only a visual linear fit and no formal statistics.
- We add: (1) formal curve fitting (log-linear vs. sigmoid + AIC), (2) quantified slopes per task, (3) item-level alignment scaling, (4) distribution-level D_KL scaling, and (5) cross-task alignment scaling.

**Data sources:** All data is fetched from the GCS bucket (`gs://levante-bench`). No new inference runs needed.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

ROOT = Path('.').resolve()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts' / 'analysis'))

import scaling_laws_alignment as sla
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

fig_dir = ROOT / 'paper' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {ROOT}')

## 1. Fetch data from GCS bucket

In [ ]:
print('Fetching accuracy data...')
acc_df = sla.fetch_accuracy_data()
print(f'  {len(acc_df)} rows ({acc_df["model"].nunique()} models)')

print('\nFetching KL divergence data...')
kl_df = sla.fetch_kl_data()
kl_agg = sla.compute_mean_kl(kl_df)
print(f'  {len(kl_df)} raw KL rows -> {len(kl_agg)} model x task cells')

print('\nFetching item-level data...')
item_df = sla.fetch_item_data()
print(f'  {len(item_df)} item rows')

print('\nFetching closest ability bin data...')
cab_df = sla.fetch_closest_ability_bin()
print(f'  {len(cab_df)} rows')

## 2. Compute alignment metrics

In [ ]:
print('Computing item-level correlations...')
corr_df = sla.compute_item_correlations(item_df, kl_df)
print(f'  {len(corr_df)} model x task correlation cells')

print('\nComputing cross-task alignment...')
ct_df = sla.compute_cross_task_alignment(acc_df)
print(f'  {len(ct_df)} models with cross-task rho')

## 3. Accuracy overview

Quick look at accuracy per model and task before fitting curves.

In [ ]:
acc_pivot = acc_df.pivot_table(index='model', columns='task_id', values='accuracy')

params_map = {m['bucket_name']: m['params_b'] for m in sla.V1_MODELS}
acc_pivot['params_b'] = acc_pivot.index.map(params_map)
acc_pivot = acc_pivot.sort_values('params_b')

display_cols = ['params_b'] + sorted(sla.TASKS)
acc_pivot[display_cols].round(3)

## 4. Fit scaling curves

For each task, we fit:
- **Log-linear:** $y = a \cdot \log_{10}(\text{params}) + b$
- **Sigmoidal:** $y = \frac{L}{1 + e^{-k(\log_{10}(\text{params}) - x_0)}} + b$

We compare them using AIC (lower = better).

In [ ]:
fit_by_task = {}
rows = []

for task in sla.TASKS:
    sub = acc_df[acc_df['task_id'] == task]
    if len(sub) < 5:
        continue
    fit_r = sla.fit_scaling(sub['params_b'].values, sub['accuracy'].values)
    fit_by_task[task] = fit_r
    
    ll = fit_r.get('loglinear')
    sig = fit_r.get('sigmoid')
    rows.append({
        'Task': sla.TASK_LABELS[task],
        'Domain': sla.TASK_DOMAIN[task],
        'Log-linear slope': ll['params']['a'] if ll else None,
        'Log-linear R2': ll['r2'] if ll else None,
        'Sigmoid R2': sig['r2'] if sig else None,
        'Best fit': fit_r['best'],
    })

fit_summary = pd.DataFrame(rows)
fit_summary.round(3)

### Key finding: Language tasks scale steeply; spatial tasks are flat

| Domain | Tasks | Slope range | R2 range |
|--------|-------|-------------|----------|
| Language | Vocab, TROG, Math, ToM | 0.14 - 0.29 | 0.53 - 0.75 |
| Spatial | Mental rotation, Matrix reasoning | 0.04 - 0.09 | 0.13 - 0.41 |

## 5. Main figure: 3-panel scaling alignment

**Panel A:** Mean accuracy vs model size (with fitted curve)  
**Panel B:** Item-level correlation (model accuracy vs human D_KL)  
**Panel C:** Mean D_KL between model and human response distributions (lower = better)

In [ ]:
agg_acc = acc_df.groupby(['model', 'params_b']).agg(mean_acc=('accuracy', 'mean')).reset_index()
fit_acc = sla.fit_scaling(agg_acc['params_b'].values, agg_acc['mean_acc'].values)

fit_item = None
if not corr_df.empty:
    agg_corr = corr_df.groupby(['model', 'params_b']).agg(mean_rpb=('r_pb', 'mean')).reset_index()
    if len(agg_corr) >= 3:
        fit_item = sla.fit_scaling(agg_corr['params_b'].values, agg_corr['mean_rpb'].values)

fit_kl = None
if not kl_agg.empty:
    agg_kl = kl_agg.groupby(['model', 'params_b']).agg(mean_dkl=('mean_dkl', 'mean')).reset_index()
    if len(agg_kl) >= 3:
        fit_kl = sla.fit_scaling(agg_kl['params_b'].values, agg_kl['mean_dkl'].values)

fit_results = {'accuracy': fit_acc, 'item_corr': fit_item, 'kl': fit_kl}

# --- Build 3-panel figure ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

if not acc_df.empty:
    agg = acc_df.groupby(['model', 'params_b', 'family']).agg(mean_acc=('accuracy', 'mean')).reset_index()
    sla._plot_scatter(axes[0], agg, 'params_b', 'mean_acc', fit_result=fit_results.get('accuracy'),
                      ylabel='Mean accuracy', title='A. Accuracy scaling')
    axes[0].set_ylim(0, 1.05)

if not corr_df.empty:
    agg = corr_df.groupby(['model', 'params_b', 'family']).agg(mean_rpb=('r_pb', 'mean')).reset_index()
    sla._plot_scatter(axes[1], agg, 'params_b', 'mean_rpb', fit_result=fit_results.get('item_corr'),
                      ylabel='Mean r_pb (item difficulty)', title='B. Item alignment scaling')

if not kl_agg.empty:
    agg = kl_agg.groupby(['model', 'params_b', 'family']).agg(mean_dkl=('mean_dkl', 'mean')).reset_index()
    sla._plot_scatter(axes[2], agg, 'params_b', 'mean_dkl', fit_result=fit_results.get('kl'),
                      ylabel='Mean D_KL', title='C. Distribution alignment scaling', invert_y=True)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(sla.FAMILY_COLORS),
           fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(fig_dir / 'scaling_alignment.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 6. Per-task accuracy scaling curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes_flat = axes.flatten()

for i, task in enumerate(sla.TASKS):
    ax = axes_flat[i]
    sub = acc_df[acc_df['task_id'] == task].copy()
    if sub.empty:
        continue

    for family, color in sla.FAMILY_COLORS.items():
        mask = sub['family'] == family
        s = sub[mask]
        if s.empty:
            continue
        marker = sla.FAMILY_MARKERS.get(family, 'o')
        ax.scatter(s['params_b'], s['accuracy'], c=color, marker=marker, s=50, alpha=0.8,
                   label=family, edgecolors='white', linewidths=0.3, zorder=3)

    fit_r = fit_by_task.get(task)
    if fit_r and fit_r.get('best'):
        x_range = np.logspace(np.log10(sub['params_b'].min() * 0.8),
                              np.log10(sub['params_b'].max() * 1.2), 200)
        best = fit_r['best']
        info = fit_r[best]
        if best == 'loglinear':
            y_pred = sla.log_linear(x_range, **info['params'])
        else:
            y_pred = sla.sigmoid(x_range, **info['params'])
        ax.plot(x_range, y_pred, 'k--', alpha=0.5, linewidth=1.2, zorder=2)
        slope_txt = f"slope={info['params']['a']:.3f}" if best == 'loglinear' else ''
        ax.text(0.03, 0.95, f"{best} R2={info['r2']:.3f}\n{slope_txt}".strip(),
                transform=ax.transAxes, fontsize=7, va='top', ha='left', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    domain = sla.TASK_DOMAIN[task]
    ax.set_xscale('log')
    ax.set_title(f"{sla.TASK_LABELS[task]} ({domain})", fontsize=10, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Parameters (B)', fontsize=8)
    ax.set_ylabel('Accuracy', fontsize=8)
    ax.grid(True, alpha=0.2)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=len(sla.FAMILY_COLORS),
           fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.01))
fig.suptitle('Accuracy scaling by task', fontsize=12, fontweight='bold')
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
fig.savefig(fig_dir / 'scaling_accuracy_by_task.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 7. Cross-task alignment vs model scale

For each model, we compute the Spearman correlation between its task-accuracy vector and the human task-accuracy vector. Does this correlation increase with model size?

In [ ]:
if not ct_df.empty:
    fig, ax = plt.subplots(figsize=(6, 4))
    for family, color in sla.FAMILY_COLORS.items():
        mask = ct_df['family'] == family
        sub = ct_df[mask]
        if sub.empty:
            continue
        marker = sla.FAMILY_MARKERS.get(family, 'o')
        ax.scatter(sub['params_b'], sub['spearman_rho'], c=color, marker=marker, s=60, alpha=0.8,
                   label=family, edgecolors='white', linewidths=0.3, zorder=3)

    if len(ct_df) >= 3:
        from scipy.optimize import curve_fit
        popt, _ = curve_fit(sla.log_linear, ct_df['params_b'].values, ct_df['spearman_rho'].values)
        x_r = np.logspace(np.log10(ct_df['params_b'].min() * 0.8),
                          np.log10(ct_df['params_b'].max() * 1.2), 200)
        ax.plot(x_r, sla.log_linear(x_r, *popt), 'k--', alpha=0.5, linewidth=1.2)
        y_pred = sla.log_linear(ct_df['params_b'].values, *popt)
        r2 = sla._r_squared(ct_df['spearman_rho'].values, y_pred)
        ax.text(0.03, 0.05, f'slope={popt[0]:.3f}, R2={r2:.3f}',
                transform=ax.transAxes, fontsize=8, va='bottom', fontfamily='monospace',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    ax.axhline(0, color='gray', linestyle=':', alpha=0.5)
    ax.set_xscale('log')
    ax.set_xlabel('Parameters (B)', fontsize=10)
    ax.set_ylabel('Spearman rho (model vs human task accuracies)', fontsize=10)
    ax.set_title('Cross-task alignment vs model scale', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    ax.grid(True, alpha=0.2)
    fig.tight_layout()
    fig.savefig(fig_dir / 'scaling_crosstask.pdf', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('No cross-task data available.')

## 8. Cross-task alignment details

In [ ]:
if not ct_df.empty:
    rho, p = stats.spearmanr(ct_df['params_b'].values, ct_df['spearman_rho'].values)
    print(f'Cross-task alignment vs log(params):')
    print(f'  Spearman rho = {rho:.3f}, p = {p:.3f}')
    print(f'  Mean cross-task rho = {ct_df["spearman_rho"].mean():.3f}')
    print()
    print(ct_df.sort_values('params_b')[['model', 'params_b', 'family', 'spearman_rho']].to_string(index=False))

## 9. Summary

### Key findings

1. **Accuracy scales steeply for language, not for spatial tasks.** Language tasks (Vocab, TROG, Math) have log-linear slopes of 0.25-0.29; spatial tasks (Mental Rotation, Matrix Reasoning) have slopes of 0.04-0.09. Sigmoidal fits are preferred by AIC, consistent with Observational Scaling Laws and Sloth.

2. **Bigger models struggle on the same items children find hard.** Item-level correlation between model accuracy and human D_KL increases weakly with model size.

3. **Bigger models make more human-like errors.** Mean D_KL between model and human response distributions decreases with scale. This extends DevBench's single-model finding to 20 models from 7 families.

4. **Bigger models agree more with children on which tasks are hard - but a gap remains.** Cross-task Spearman rho increases with scale (rho=0.475, p=0.034), but even 200B models find spatial tasks disproportionately hard relative to language tasks.

### One-sentence takeaway

> Scaling VLMs makes them more human-like on language tasks, but does not teach them to think spatially like an 8-year-old.